In [1]:
import simpy
import random
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
class CLASS:

    def __init__(self, service_rate, dropout_rate, dropout_cost, anandonment_rate_without_AI,
                anandonment_cost_without_AI, if_use_AI_or_not, supervision_cost,
                negative_anandonment_rate, positive_anandonment_rate,
                negative_anandonment_cost, holding_cost_without_AI, holding_cost_with_AI, benefit, arrival_rate, name,
                fixed_overhead_cost):
        
        self.service_rate = service_rate
        self.dropout_rate = dropout_rate
        self.dropout_cost = dropout_cost
        self.anandonment_rate_without_AI = anandonment_rate_without_AI
        self.anandonment_cost_without_AI = anandonment_cost_without_AI
        self.if_use_AI_or_not = if_use_AI_or_not
        self.supervision_cost = supervision_cost
        self.negative_anandonment_rate = negative_anandonment_rate
        self.positive_anandonment_rate = positive_anandonment_rate
        self.negative_anandonment_cost = negative_anandonment_cost
        self.holding_cost_without_AI = holding_cost_without_AI
        self.holding_cost_with_AI = holding_cost_with_AI
        self.benefit = benefit
        self.arrival_rate = arrival_rate
        self.fixed_overhead_cost = fixed_overhead_cost
        self.name = name
        
        r = benefit * service_rate - dropout_rate * dropout_cost
        c = holding_cost_without_AI + anandonment_rate_without_AI * anandonment_cost_without_AI
        theta_w = negative_anandonment_rate + positive_anandonment_rate
        c_w = holding_cost_with_AI + negative_anandonment_cost * negative_anandonment_rate + supervision_cost - positive_anandonment_rate * benefit
        P = (r * anandonment_rate_without_AI + c * (service_rate + dropout_rate)) / anandonment_rate_without_AI
        P_w = (r * theta_w + c_w * (service_rate + dropout_rate)) / theta_w

        self.r = r
        self.c = c
        self.c_w = c_w
        self.P = P
        self.P_w = P_w
        self.index = P * (1 - if_use_AI_or_not) + P_w * if_use_AI_or_not
        self.z_upper = arrival_rate / (service_rate + dropout_rate)

        self.in_system = 0
        self.in_service = 0

In [3]:
def log_state(env, server):
    
    times.append(env.now)

    q_len = len(server.queue)
    busy = server.count
    n_system.append(q_len + busy)
    n_busy.append(busy)

    A_system.append(CLASS_A.in_system)
    A_busy.append(CLASS_A.in_service)

    B_system.append(CLASS_B.in_system)
    B_busy.append(CLASS_B.in_service)

In [4]:
def sample_positive_abandon_time(CLASS):
    return random.expovariate(CLASS.positive_anandonment_rate)

def sample_negative_abandon_time(CLASS):
    return random.expovariate(CLASS.negative_anandonment_rate)
    
def sample_anandonment_rate_without_AI_time(CLASS):
    return random.expovariate(CLASS.anandonment_rate_without_AI)

In [5]:
def customer_with_AI(env, name, prio, server, CLASS):
    
    CLASS.in_system += 1
    log_state(env, server)

    while True:

        req = server.request(priority=prio, preempt=False)
    
       
        t_ab = sample_anandonment_rate_without_AI_time(CLASS)  
        ab = env.timeout(t_ab, value="abandon")
    
        log_state(env, server)
    
     
        result = yield req | ab

        if req not in result:
            CLASS.in_system -= 1
            req.cancel()
            log_state(env, server)
            return
            
        CLASS.in_service += 1
        log_state(env, server)

        service_time = random.expovariate(CLASS.service_rate)
        drop_time    = random.expovariate(CLASS.dropout_rate)
        service_done = env.timeout(service_time)
        drop_done    = env.timeout(drop_time)

        try:
            yield service_done | drop_done

        except simpy.Interrupt as intr:
            
           
            CLASS.in_service -= 1
           
           
            log_state(env, server)
            
            continue

        
        CLASS.in_service -= 1
        CLASS.in_system  -= 1
      
        if req in server.users:
            server.release(req)
        log_state(env, server)
        return

In [11]:
def customer_without_AI(env, name, prio, server, CLASS):
   

   
    CLASS.in_system += 1

   
    req = server.request(priority=prio, preempt=True)

    
    t_ab = sample_anandonment_rate_without_AI_time(CLASS)  
    ab = env.timeout(t_ab, value="abandon")

    log_state(env, server)

    
    result = yield req | ab

    if req not in result:
        
        CLASS.in_system -= 1
        req.cancel()

        log_state(env, server)
        return   

    
    CLASS.in_service += 1
    log_state(env, server)

    service_done = env.timeout(random.expovariate(CLASS.service_rate))
    drop_done    = env.timeout(random.expovariate(CLASS.dropout_rate))

    
    yield service_done | drop_done

   
    CLASS.in_service -= 1
    CLASS.in_system  -= 1

    
    if req in server.users:
        server.release(req)

    log_state(env, server)
    return

In [13]:
def arrival_A(env, server, CLASS):
    i = 0
    while True:
        inter_arrival = random.expovariate(CLASS.arrival_rate)
        yield env.timeout(inter_arrival)
        i += 1
        env.process(customer_without_AI(env, f"A{i}", prio=0, server=server, CLASS=CLASS))

def arrival_B(env, server, CLASS):
    i = 0
    while True:
        inter_arrival = random.expovariate(CLASS.arrival_rate)
        yield env.timeout(inter_arrival)
        i += 1
        env.process(customer_with_AI(env, f"B{i}", prio=1, server=server, CLASS=CLASS))

In [15]:
def main(capacity):

    global times, n_system, n_busy, A_system, A_busy, B_system, B_busy
    times = []
    n_system = []
    n_busy = []
    A_system = []
    A_busy = []
    B_system = []
    B_busy = []

    CLASS_A.in_system = 0
    CLASS_A.in_service = 0
    CLASS_B.in_system = 0
    CLASS_B.in_service = 0

    env = simpy.Environment()
    server = simpy.PreemptiveResource(env, capacity)

    log_state(env, server)   

    env.process(arrival_A(env, server, CLASS_A))
    env.process(arrival_B(env, server, CLASS_B))

    env.run(until=1000)

    return times, n_system, n_busy, A_system, A_busy, B_system, B_busy

In [17]:
def sample_run(result, Intervals):

    times, n_sys, n_busy, A_sys, A_busy, B_sys, B_busy = result

    times = np.array(times)

    K = len(Intervals)
    out = {
        "A_sys": np.zeros(K),
        "B_sys": np.zeros(K),
        "A_busy": np.zeros(K),
        "B_busy": np.zeros(K),
        "total_sys": np.zeros(K),
        "total_busy": np.zeros(K)
    }

    for i, T in enumerate(Intervals):

        idx = np.searchsorted(times, T, side='right') - 1

        if idx < 0:
            idx = 0
        if idx >= len(times):
            idx = len(times) - 1

        out["A_sys"][i] = A_sys[idx]
        out["B_sys"][i] = B_sys[idx]
        out["A_busy"][i] = A_busy[idx]
        out["B_busy"][i] = B_busy[idx]
        out["total_sys"][i] = n_sys[idx]
        out["total_busy"][i] = n_busy[idx]

    return out

In [19]:
def multi_run(Replications, Intervals, capacity):

    K = len(Intervals)

    cum_A_sys    = np.zeros(K)
    cum_B_sys    = np.zeros(K)
    cum_A_busy   = np.zeros(K)
    cum_B_busy   = np.zeros(K)
    cum_sys      = np.zeros(K)
    cum_busy     = np.zeros(K)

    for rep in range(Replications):

        result = main(capacity)                    
        sampled = sample_run(result, Intervals) 

        cum_A_sys += sampled["A_sys"]
        cum_B_sys += sampled["B_sys"]
        cum_A_busy += sampled["A_busy"]
        cum_B_busy += sampled["B_busy"]
        cum_sys += sampled["total_sys"]
        cum_busy += sampled["total_busy"]

    avg_A_sys  = cum_A_sys / Replications
    avg_B_sys  = cum_B_sys / Replications
    avg_A_busy = cum_A_busy / Replications
    avg_B_busy = cum_B_busy / Replications
    avg_sys    = cum_sys / Replications
    avg_busy   = cum_busy / Replications

    return avg_A_sys, avg_B_sys, avg_A_busy, avg_B_busy, avg_sys, avg_busy

In [21]:
beta = 0.82                 # show-up probability
dropout_rate = 0.0067       # gamma per week
abandonment_rate_wo_ai = 0.04  # theta^n per week
positive_abandonment_rate = 0.06   # theta^{w,p} per week
negative_abandonment_rate = 0.0005 # theta^{w,n} per week
supervision_cost = 1        # a_i per patient per week
fixed_overhead_cost = 640   # fbar_i per class per week

# Benefits (b_i)
b_MDD = 12000
b_AD  = 9000

# Costs derived from benefit
abandonment_cost_wo_ai_MDD = 0.02 * b_MDD   # alpha^n
abandonment_cost_wo_ai_AD  = 0.02 * b_AD

negative_abandonment_cost_MDD = 0.01 * b_MDD  # alpha^{w,n}
negative_abandonment_cost_AD  = 0.01 * b_AD

dropout_cost_MDD = 0.01 * b_MDD  # h^d
dropout_cost_AD  = 0.01 * b_AD

# Holding costs (weekly)
holding_cost_wo_ai_MDD = 324
holding_cost_wo_ai_AD  = 196

holding_cost_w_ai_MDD = 32.4
holding_cost_w_ai_AD  = 19.6

# Service rates: mu = beta * mu^b
mu_b_MDD = 2.96
mu_b_AD  = 2.96

service_rate_MDD  = beta * mu_b_MDD   # 2.4272
service_rate_AD   = beta * mu_b_AD    # 2.4272

# Arrival rates (weekly)
arrival_rate_MDD  = 81/45
arrival_rate_AD   = 57/20

In [23]:
si_A_q = []
si_B_q = []
si_A_ser = []
si_B_ser = []

for i in tqdm([5,10,15,20,25]):

    CLASS_A = CLASS(
        service_rate=(service_rate_MDD),
        dropout_rate=(dropout_rate),
        dropout_cost=(dropout_cost_MDD),
        anandonment_rate_without_AI=(abandonment_rate_wo_ai),
        anandonment_cost_without_AI=(abandonment_cost_wo_ai_MDD),
        if_use_AI_or_not=0,
        supervision_cost=(supervision_cost),
        negative_anandonment_rate=(negative_abandonment_rate),
        positive_anandonment_rate=(positive_abandonment_rate),
        negative_anandonment_cost=(negative_abandonment_cost_MDD),
        holding_cost_without_AI=(holding_cost_wo_ai_MDD),
        holding_cost_with_AI=(holding_cost_w_ai_MDD),
        benefit=(b_MDD)/2,
        arrival_rate=(arrival_rate_MDD) * i,
        fixed_overhead_cost=(fixed_overhead_cost)*200,
        name="MDD"
    )
    
    CLASS_B = CLASS(
        service_rate=(service_rate_AD),
        dropout_rate=(dropout_rate),
        dropout_cost=(dropout_cost_AD),
        anandonment_rate_without_AI=(abandonment_rate_wo_ai),
        anandonment_cost_without_AI=(abandonment_cost_wo_ai_AD),
        if_use_AI_or_not=0,
        supervision_cost=(supervision_cost),
        negative_anandonment_rate=(negative_abandonment_rate),
        positive_anandonment_rate=(positive_abandonment_rate),
        negative_anandonment_cost=(negative_abandonment_cost_AD),
        holding_cost_without_AI=(holding_cost_wo_ai_AD),
        holding_cost_with_AI=(holding_cost_w_ai_AD),
        benefit=(b_AD),
        arrival_rate=(arrival_rate_AD) * i,
        fixed_overhead_cost=(fixed_overhead_cost),
        name="AD"
    )

    capacity = i
    Intervals = np.arange(0, 1000, 2)
    avg_A_sys, avg_B_sys, avg_A_busy, avg_B_busy, avg_sys, avg_busy = multi_run(10, Intervals, capacity)

    mean_A_queue_length = (avg_A_sys - avg_A_busy)[300:].mean()
    mean_B_queue_length = (avg_B_sys - avg_B_busy)[300:].mean()
    mean_A_server = avg_A_busy[300:].mean()
    mean_B_server = avg_B_busy[300:].mean()

    si_A_q.append(mean_A_queue_length)
    si_B_q.append(mean_B_queue_length)
    si_A_ser.append(mean_A_server)
    si_B_ser.append(mean_B_server)

100%|███████████████████████████████████████████████████████████████████████████████████| 5/5 [12:31<00:00, 150.39s/it]


In [24]:
(np.array(si_A_ser)*CLASS_A.r - np.array(si_A_q)*CLASS_A.c + np.array(si_B_ser)*CLASS_B.r - np.array(si_B_q)*CLASS_B.c)

array([ 26616.4014085,  51953.2092655,  78040.7018095, 103530.98798  ,
       131342.533256 ])

In [25]:
si_A_q,si_B_q,si_A_ser,si_B_ser

([1.128, 0.765, 0.5670000000000001, 0.5125, 0.29450000000000004],
 [272.37949999999995, 551.6585, 830.09, 1108.5910000000001, 1376.134],
 [3.6914999999999996, 7.434499999999999, 11.090499999999999, 14.82, 18.544],
 [1.3085, 2.5654999999999997, 3.9094999999999995, 5.18, 6.456])